# GuarantorLens - additional models

The core tool scores a loan's default risk with a supervised **classifier** (XGBoost) in `train.ipynb`.
This notebook adds **three more model families**, each answering a *different* question, so the system
is a multi-model tool rather than "just classification". Everything here is kept deliberately simple and
uses only `networkx`, `scikit-learn`, `matplotlib` (all pre-installed on Colab).

| Model | Question it answers | Where it fits in the tool |
|---|---|---|
| **Network: communities + contagion** | Which guarantee groups are risky, and how far does one default spread? | Network / Contagion / Weak-links |
| **Survival analysis** | *How long* does a loan last before it is written off? | Monitoring / portfolio |
| **Clustering + anomaly** | What borrower *types* exist, and which applications are unusual? | Insights / Assess |

Run this **after** `train.ipynb` (it reads the exported `guarantorlens_loans.json` and
`guarantorlens_members.json`).

In [ ]:
import os, json, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
plt.rcParams.update({"font.size": 11, "figure.dpi": 120, "savefig.bbox": "tight"})

# find the exported tables (works locally and on Colab after train.ipynb)
def _find(name):
    for p in [name, f"models/guarantorlens_new/{name}", f"app/artifacts/{name}"]:
        if os.path.exists(p): return p
    raise FileNotFoundError(f"{name} not found - run train.ipynb first, or upload it.")
loans = json.load(open(_find("guarantorlens_loans.json")))
members = json.load(open(_find("guarantorlens_members.json")))
M = {m["member_id"]: m for m in members}
L = pd.DataFrame(loans)
L["disb"] = pd.to_datetime(L["disb_date"], errors="coerce")
OUT = "models/guarantorlens_new"; os.makedirs(OUT, exist_ok=True)
print(f"loans {len(L)}  written-off {int((L.label==1).sum())} ({L.label.mean():.1%})  members {len(M)}")

## 1. The guarantee network - communities and contagion

A guarantee network is a **graph**: every loan links a borrower to the members who guarantee it. Here we
study that graph itself, not the borrower's finances.

- **Community detection (Louvain method):** automatically finds tight-knit groups - clusters of people who
  guarantee one another. `modularity` (0-1) says how cleanly the network splits into groups. We then measure
  the **write-off rate inside each group**; some groups are far riskier than the portfolio average.
- **Contagion:** if a member is written off, every loan they guarantee is **exposed**. We follow that exposure
  step by step ("rounds") to see how far trouble can spread through the network.

In [ ]:
G = nx.Graph()
for _, r in L.iterrows():
    for g in (r["guarantors"] or []):
        G.add_edge(r["borrower"], g)
print(f"graph: {G.number_of_nodes():,} members (nodes), {G.number_of_edges():,} guarantee links (edges)")

comms = nx.community.louvain_communities(G, seed=42)
mod = nx.community.modularity(G, comms)
print(f"Louvain found {len(comms):,} communities  |  modularity = {mod:.3f} (higher = cleaner groups)")

node_comm = {n: i for i, c in enumerate(comms) for n in c}
L["comm"] = L["borrower"].map(node_comm)
crate = L.groupby("comm").agg(loans=("label","size"), bad=("label","sum"))
crate["bad_rate"] = crate.bad / crate.loans
crate["members"] = [len(comms[int(i)]) for i in crate.index]
crate.sort_values("bad_rate", ascending=False).to_csv(f"{OUT}/10_communities.csv")
big = crate[crate.loans >= 30].sort_values("bad_rate", ascending=False).head(10)
print("\nRiskiest guarantee communities (>=30 loans):")
print(big[["members","loans","bad","bad_rate"]].round(3).to_string())

fig, ax = plt.subplots(figsize=(8,4))
ax.bar(range(len(big)), big.bad_rate, color="#c0392b")
ax.axhline(L.label.mean(), ls="--", c="grey", label=f"portfolio {L.label.mean():.1%}")
ax.set_xticks(range(len(big))); ax.set_xticklabels([f"C{int(i)}" for i in big.index])
ax.set_ylabel("write-off rate"); ax.set_title("Riskiest guarantee communities"); ax.legend()
plt.savefig(f"{OUT}/10_community_default.png"); plt.show()

# contagion: seed = written-off members, spread to anyone whose loan they guarantee
defaulters = {mid for mid, m in M.items() if m.get("ever_defaulted") == 1}
lg = [(r["borrower"], set(r["guarantors"] or []), r["amount"]) for _, r in L.iterrows()]
exp = [(b, amt) for b, gs, amt in lg if gs & defaulters]
print(f"\n1-hop exposure: {len(exp):,} loans are guaranteed by a written-off member, "
      f"RWF {sum(a for _, a in exp):,.0f} at stake")
compromised = set(defaulters); reach = [len(compromised)]
for _ in range(4):
    grew = {b for b, gs, amt in lg if gs & compromised}
    if compromised >= grew: break
    compromised |= grew; reach.append(len(compromised))
print("cascade reach per round:", reach)
fig, ax = plt.subplots(figsize=(7,3.8))
ax.plot(range(len(reach)), reach, "o-", color="#173C8E")
ax.set_xlabel("propagation round"); ax.set_ylabel("members touched")
ax.set_title("How far a default can spread through the network"); plt.savefig(f"{OUT}/10_contagion.png"); plt.show()

# picture of one risky community (capped for readability)
sub = list(comms[int(big.index[0])])[:60]; Gs = G.subgraph(sub)
fig, ax = plt.subplots(figsize=(7,6))
cols = ["#c0392b" if n in defaulters else "#173C8E" for n in Gs.nodes()]
nx.draw_networkx(Gs, nx.spring_layout(Gs, seed=42, k=0.5), ax=ax, node_size=90,
                 node_color=cols, with_labels=False, edge_color="#ccc", width=0.6)
ax.set_title("A guarantee community (red = written off)"); ax.axis("off")
plt.savefig(f"{OUT}/10_network_sample.png"); plt.show()

## 2. Survival analysis - time to default

Classification asks *"will this loan default?"*. Survival analysis asks *"how long does a loan last before
it is written off?"* - the timing, not just the yes/no.

- The **Kaplan-Meier curve** shows the share of loans **still performing** as the months pass. A loan that is
  repaid or still active is *censored*: it simply leaves the calculation when we stop watching it.
- We split by **loan size** to check whether larger loans fail sooner.

*Data note:* the dataset records each loan's **disbursement date** and its **final outcome**, not the exact
write-off date, so we approximate a loan's age as months from disbursement to a fixed observation date. This
is a standard "months-on-book" proxy.

In [ ]:
OBS = pd.Timestamp("2024-12-31")
L["months"] = ((OBS - L["disb"]).dt.days / 30.44).round().clip(lower=1)
L["event"]  = (L["label"] == 1).astype(int)   # 1 = written off (the event); 0 = censored

def km(df):                                    # Kaplan-Meier by hand (no extra library)
    S, rows = 1.0, []
    for t in np.sort(df["months"].unique()):
        at_risk = (df["months"] >= t).sum()
        d = ((df["months"] == t) & (df["event"] == 1)).sum()
        if at_risk: S *= (1 - d / at_risk)
        rows.append((t, S))
    return pd.DataFrame(rows, columns=["month", "survival"])

def s_at(k, m):
    r = k[k.month <= m]; return round(float(r.survival.iloc[-1]), 4) if len(r) else 1.0

overall = km(L)
L["amt_tier"] = pd.qcut(L["amount"], 3, labels=["small","medium","large"])
fig, ax = plt.subplots(figsize=(8,4.6))
ax.step(overall.month, overall.survival, where="post", color="black", lw=2, label="all loans")
tbl = [{"group":"all", "survival_12mo":s_at(overall,12), "survival_24mo":s_at(overall,24)}]
for tier, c in [("small","#2ecc71"),("medium","#f39c12"),("large","#c0392b")]:
    k = km(L[L.amt_tier == tier]); ax.step(k.month, k.survival, where="post", color=c, label=f"{tier} loan")
    tbl.append({"group":tier, "survival_12mo":s_at(k,12), "survival_24mo":s_at(k,24)})
ax.set_xlabel("months on book"); ax.set_ylabel("share still performing"); ax.set_ylim(0.9, 1.001)
ax.set_title("Loan survival over time (Kaplan-Meier)"); ax.legend(); plt.savefig(f"{OUT}/11_survival_km.png"); plt.show()
pd.DataFrame(tbl).to_csv(f"{OUT}/11_survival_table.csv", index=False)
print(pd.DataFrame(tbl).to_string(index=False))
print("\nReading it: ~98% of loans are still performing at 24 months; large loans fail soonest.")

## 3. Borrower segmentation + anomaly flag

- **Clustering (KMeans)** groups borrowers into a few *types* from their loan size, savings, salary and number
  of guarantors - **without** using the default label. We then read off each type's write-off rate. The
  `silhouette` score (0-1) says how well-separated the groups are.
- **Anomaly detection (Isolation Forest)** scores how *unusual* each application is. Flagging the most unusual
  10% catches loans that default several times more often than average - a cheap first screen for the officer.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest

L["savings"] = L["borrower"].map(lambda b: M.get(b,{}).get("savings")).clip(lower=0)
L["salary"]  = L["borrower"].map(lambda b: M.get(b,{}).get("salary")).clip(lower=0)
L["n_guar"]  = L["guarantors"].map(lambda g: len(g or []))
L["loan_to_sav"] = L["amount"] / (L["savings"].fillna(0) + 1)
FEATS = ["amount","savings","salary","loan_to_sav","n_guar"]
X = SimpleImputer(strategy="median").fit_transform(L[FEATS])
X = np.clip(X, np.nanpercentile(X,1,0), np.nanpercentile(X,99,0))   # tame extreme values
Xs = StandardScaler().fit_transform(X)

km4 = KMeans(n_clusters=4, random_state=42, n_init=10).fit(Xs)
L["cluster"] = km4.labels_
print(f"KMeans k=4  |  silhouette = {silhouette_score(Xs, km4.labels_):.3f}")
prof = L.groupby("cluster").agg(loans=("label","size"), bad_rate=("label","mean"),
        avg_amount=("amount","mean"), median_savings=("savings","median"), avg_guarantors=("n_guar","mean"))
prof.round(3).to_csv(f"{OUT}/12_clusters.csv"); print(prof.round(3).to_string())

pca = PCA(2).fit_transform(Xs)
fig, ax = plt.subplots(figsize=(7,5.2))
ax.scatter(pca[:,0], pca[:,1], c=L.cluster, cmap="tab10", s=6, alpha=0.5)
ax.set_title("Borrower segments (KMeans, PCA view)"); ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
plt.savefig(f"{OUT}/12_cluster_scatter.png"); plt.show()

iso = IsolationForest(n_estimators=200, contamination=float(L.label.mean()), random_state=42).fit(Xs)
L["anomaly"] = -iso.score_samples(Xs)
flag = L[L.anomaly >= L.anomaly.quantile(0.90)]
print(f"\nAnomaly flag (most unusual 10%): write-off rate {flag.label.mean():.1%} "
      f"vs portfolio {L.label.mean():.1%}  ->  {flag.label.mean()/L.label.mean():.1f}x more likely to default")

## Summary

GuarantorLens now uses a **portfolio of models**, each suited to its task:

1. **Classification** (`train.ipynb`) - scores default risk for a new loan.
2. **Network model** - Louvain communities + contagion on the guarantee graph; finds risky groups and traces exposure.
3. **Survival analysis** - Kaplan-Meier; estimates how long loans last, and that large loans fail soonest.
4. **Clustering + anomaly** - borrower segments, plus an unusual-application flag (~3x default lift).

All artifacts are saved in `models/guarantorlens_new/` (files `10_*`, `11_*`, `12_*`) for the report.